In [1]:
from llama_cpp import Llama
import json
import re
import logging
import sqlparse
from datetime import datetime

In [2]:
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler("mistral_trace.log", encoding="utf-8"),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)


In [3]:
llm = Llama(
    model_path=r"C:\Users\HP\Documents\GINF2\StagePfa_INVOLYS\Model\mistral-7b-instruct-v0.1.Q4_K_M.gguf",
    n_ctx=8192,
    n_threads=8,
    verbose=False
)

llama_context: n_ctx_per_seq (8192) < n_ctx_train (32768) -- the full capacity of the model will not be utilized


In [4]:
# 📊 Schéma simulé
erp_schema = {
    "tables": {
        "commandes": {
            "columns": {
                "id": "INT PRIMARY KEY",
                "date_commande": "DATE",
                "montant": "DECIMAL(10,2)",
                "client_id": "INT FOREIGN KEY",
                "produit_id": "INT FOREIGN KEY"
            },
            "description": "Commandes passées par les clients"
        },
        "clients": {
            "columns": {
                "id": "INT PRIMARY KEY",
                "nom": "VARCHAR(100)",
                "secteur": "VARCHAR(50)",
                "ville": "VARCHAR(50)"
            },
            "description": "Informations clients"
        },
        "produits": {
            "columns": {
                "id": "INT PRIMARY KEY",
                "nom": "VARCHAR(100)",
                "categorie": "VARCHAR(50)",
                "prix_unitaire": "DECIMAL(10,2)"
            },
            "description": "Catalogue produits"
        },
        "fournisseurs": {
            "columns": {
                "id": "INT PRIMARY KEY",
                "nom": "VARCHAR(100)",
                "pays": "VARCHAR(50)",
                "categorie_fournisseur": "VARCHAR(50)"
            },
            "description": "Fournisseurs disponibles"
        }
    },
    "relations": {
        "commandes.client_id": "clients.id",
        "commandes.produit_id": "produits.id"
    }
}


In [5]:
def format_schema_detailed(schema):
    tables_desc = []
    for table, info in schema["tables"].items():
        cols = ", ".join([f"{col} ({type_col})" for col, type_col in info["columns"].items()])
        tables_desc.append(f"Table {table}: {cols} - {info['description']}")
    
    relations = "\nRelations:\n" + "\n".join([f"- {rel}" for rel in schema["relations"]])
    return "\n".join(tables_desc) + relations

In [ ]:
def generate_prompt(question: str, schema: dict) -> str:
    return f"""
Tu es un moteur d’analyse linguistique expert intégré à un système ERP. Ta tâche est d’analyser la question ci-dessous et d’en extraire toutes les informations nécessaires à la génération d’une requête SQL. 

QUESTION UTILISATEUR :
"{question}"

SCHÉMA DE LA BASE DE DONNÉES :
{format_schema_detailed(schema)}

OBJECTIF :
Analyse la question et produis une sortie **au format texte structuré strict**, contenant toutes les sections obligatoires, même si certaines sont vides. N’invente jamais de table ou colonne non mentionnée dans le schéma.
et  - N'inclure QUE les colonnes strictement nécessaires pour répondre à la question
   -  Exclure les colonnes techniques (ex: `id`) sauf si explicitement demandées
   -  Faire la meilleur extrcation possible des données
   -  N'ajoute aucun texte explicatif, uniquement les sections demandées
   -  Respecte **strictement** le format suivant pour la sortie, sans ajouter de texte supplémentaire ni explications 
FORMAT À RESPECTER :

INTENTION: ...
TABLES: [...]
COLONNES: [...]
FILTRES: [...] (si ils existent)
JOINTURES: [...] (si elles existent)
AGRÉGATION: ... (si elle existe)

RÈGLES :
- Ne saute **aucune section**
- Respecte **strictement** les noms du schéma
- Pour les dates : `EXTRACT(MONTH...)`, `EXTRACT(YEAR...)`
- Valeurs textuelles entourées de `'...'`

EXEMPLE :
INTENTION: SELECT  
TABLES: [commandes]  
COLONNES: [commandes.id, commandes.date_commande]  
FILTRES: [EXTRACT(MONTH FROM commandes.date_commande) BETWEEN 1 AND 3, EXTRACT(YEAR FROM commandes.date_commande) = 2024]  
JOINTURES: []  
AGRÉGATION:

Essaie d'utilisaer bcp ton analyse tu as le shéma tu as tous analse bien la réquete et donne les meilleur extrcation et les plus adequats.
""".strip()

In [7]:
def analyser_question_mistral(question):
    prompt = generate_prompt(question, erp_schema)
    response = llm.create_chat_completion(
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0.1,
        max_tokens=1024,
        stop=["</s>"]
    )
    return response['choices'][0]['message']['content']

In [8]:
def appeler_llm(prompt, stop=[], max_tokens=1024, temperature=0.1):
    try:
        result = llm(
            prompt=prompt, 
            stop=stop, 
            temperature=temperature, 
            max_tokens=max_tokens,
            top_p=0.95,
            repeat_penalty=1.1
        )
        return result["choices"][0]["text"].strip()
    except Exception as e:
        logger.error(f"Erreur LLM: {e}\nPrompt utilisé:\n{prompt}")
        return ""

In [9]:
import re

def parse_mistral_response_to_json(response_text: str) -> dict:
    result = {
        "intention": "",
        "tables": [],
        "colonnes": [],
        "filtres": [],
        "jointures": [],
        "agregation": ""
    }

    # Nettoyage : enlever les espaces en début de ligne, etc.
    response_text = response_text.strip()

    # Expression régulière globale : capture chaque section et son contenu
    pattern = r"""
        INTENTION:\s*(.*?)\n
        TABLES:\s*\[(.*?)\]\n
        COLONNES:\s*\[(.*?)\]\n
        FILTRES:\s*\[(.*?)\]\n
        JOINTURES:\s*\[(.*?)\]\n
        AGRÉGATION:\s*(.*)
    """

    match = re.search(pattern, response_text, re.DOTALL | re.VERBOSE)
    if not match:
        raise ValueError("❌ Le texte ne suit pas le format attendu")

    result["intention"] = match.group(1).strip()
    result["tables"] = [t.strip() for t in match.group(2).split(',') if t.strip()]
    result["colonnes"] = [c.strip() for c in match.group(3).split(',') if c.strip()]
    result["filtres"] = [f.strip() for f in re.split(r',(?![^(]*\))', match.group(4)) if f.strip()]
    result["jointures"] = [j.strip() for j in match.group(5).split(',') if j.strip()]
    result["agregation"] = match.group(6).strip()

    return result


In [10]:
# question_test = "Quels clients ont commandé des produits en avril 2024 ?"
# reponse = analyser_question_mistral(question_test)
# print("🧠 Réponse NLP via Mistral :\n")
# print(reponse)

In [11]:
# parsed = parse_mistral_response_to_json(reponse)
# print(json.dumps(parsed, indent=2, ensure_ascii=False))

In [ ]:
questions = [
    "Quels clients ont commandé des produits en avril 2024 ?",
    "Quelle est la commande la plus récente ?",
    "Donne-moi les produits commandés par le client Involys.",
    "Combien de commandes ont été passées en 2023 ?",
    "Quels sont les clients situés à Casablanca ?",
    "Quels sont les produits dans la catégorie informatique ?",
    "Liste les commandes de plus de 1000 MAD.",
    "Quels clients n'ont jamais commandé ?",
    "Quelles sont les villes avec le plus de commandes ?",
    "Combien de commandes ont été faites par client ?",
    "Donne-moi les montants totaux par secteur client.",
    "Quels produits ont été commandés en janvier 2024 ?",
    "Quels sont les fournisseurs situés en France ?"
]

for idx, q in enumerate(questions, 1):
    try:
        logger.info(f"📤 Question {idx}: {q}")
        reponse = analyser_question_mistral(q)
        
        logger.debug(f"🧠 Réponse brute Mistral:\n{reponse}")
        logger.info(f"🧠 Réponse NLP reçue (question {idx}):\n{reponse}")

        parsed = parse_mistral_response_to_json(reponse)
        
        logger.info(f"✅ Question {idx} analysée avec succès.")
        logger.debug(f"📦 JSON structuré (question {idx}):\n{json.dumps(parsed, indent=2, ensure_ascii=False)}")

        print(f"\n🔢 Question {idx}")
        print(f"❓ {q}")
        print(f"📦 Réponse Mistral :\n{json.dumps(parsed, indent=2, ensure_ascii=False)}")

    except Exception as e:
        logger.error(f"❌ Erreur pour la question {idx} : {q} - {str(e)}")
        print(f"\n❌ Erreur pour la question {idx} : {q}")
        print(e)



2025-07-07 11:47:49,299 - INFO - 📤 Question 1: Quels clients ont commandé des produits en avril 2024 ?
2025-07-07 11:49:23,184 - INFO - ✅ Question 1 analysée avec succès.
2025-07-07 11:49:23,189 - INFO - 📤 Question 2: Quelle est la commande la plus récente ?



🔢 Question 1
❓ Quels clients ont commandé des produits en avril 2024 ?
📦 Réponse Mistral :
{
  "intention": "SELECT",
  "tables": [
    "commandes",
    "clients"
  ],
  "colonnes": [
    "commandes.id",
    "commandes.date_commande",
    "clients.nom"
  ],
  "filtres": [
    "EXTRACT(MONTH FROM commandes.date_commande) BETWEEN 1 AND 3",
    "EXTRACT(YEAR FROM commandes.date_commande) = 2024",
    "clients.id IN (SELECT commandes.client_id FROM commandes WHERE EXTRACT(MONTH FROM commandes.date_commande) BETWEEN 1 AND 3 AND EXTRACT(YEAR FROM commandes.date_commande) = 2024)"
  ],
  "jointures": [
    "INNER JOIN commandes ON clients.id = commandes.client_id"
  ],
  "agregation": ""
}


2025-07-07 11:51:16,358 - INFO - ✅ Question 2 analysée avec succès.
2025-07-07 11:51:16,358 - INFO - 📤 Question 3: Donne-moi les produits commandés par le client Involys.



🔢 Question 2
❓ Quelle est la commande la plus récente ?
📦 Réponse Mistral :
{
  "intention": "SELECT",
  "tables": [
    "commandes"
  ],
  "colonnes": [
    "commandes.id",
    "commandes.date_commande",
    "commandes.montant",
    "clients.nom",
    "clients.secteur",
    "clients.ville",
    "produits.nom",
    "produits.categorie",
    "produits.prix_unitaire",
    "fournisseurs.nom",
    "fournisseurs.pays",
    "fournisseurs.categorie_fournisseur"
  ],
  "filtres": [
    "commandes.date_commande = (SELECT MAX(commandes.date_commande) FROM commandes)"
  ],
  "jointures": [
    "commandes.client_id = clients.id",
    "commandes.produit_id = produits.id"
  ],
  "agregation": "[commandes.id, commandes.date_commande, commandes.montant, clients.nom, clients.secteur, clients.ville, produits.nom, produits.categorie, produits.prix_unitaire, fournisseurs.nom, fournisseurs.pays, fournisseurs.categorie_fournisseur]\n\nNote: The above response is based on the assumption that the question is

2025-07-07 11:52:32,077 - INFO - ✅ Question 3 analysée avec succès.
2025-07-07 11:52:32,079 - INFO - 📤 Question 4: Combien de commandes ont été passées en 2023 ?


...

2025-07-07 11:54:51,797 - INFO - ✅ Question 5 analysée avec succès.
2025-07-07 11:54:51,799 - INFO - 📤 Question 6: Quels sont les produits dans la catégorie informatique ?



🔢 Question 5
❓ Quels sont les clients situés à Casablanca ?
📦 Réponse Mistral :
{
  "intention": "SELECT",
  "tables": [
    "clients"
  ],
  "colonnes": [
    "clients.nom",
    "clients.ville"
  ],
  "filtres": [
    "clients.ville = 'Casablanca'"
  ],
  "jointures": [],
  "agregation": "The intention of the question is to retrieve the names and cities of clients located in Casablanca. The tables involved in this query are clients, commandes, and produits. The columns required for this query are clients.nom and clients.ville. The filter applied to the query is clients.ville = 'Casablanca'. There are no join conditions or aggregations required for this query."
}


2025-07-07 11:55:40,677 - INFO - ✅ Question 6 analysée avec succès.
2025-07-07 11:55:40,680 - INFO - 📤 Question 7: Liste les commandes de plus de 1000 MAD.



🔢 Question 6
❓ Quels sont les produits dans la catégorie informatique ?
📦 Réponse Mistral :
{
  "intention": "SELECT",
  "tables": [
    "produits"
  ],
  "colonnes": [
    "produits.nom"
  ],
  "filtres": [
    "produits.categorie = 'informatique'"
  ],
  "jointures": [],
  "agregation": ""
}


KeyboardInterrupt: 